# SageHybridNets

Launch the road-only HybridNets SageMaker run from this repo using `TrainingScript.py`.

Set `RUN_MODE` to `"smoke"` for a fast sanity check with a tiny S3 subset, or `"real"` for the full training run.

Expected SageMaker input channels by mode:

- `smoke`: `s3://autonomous-bike/data/hybridnets_smoke/images/` and `s3://autonomous-bike/data/hybridnets_smoke/hybridNets_data_smoke.tar`
- `real`: `s3://autonomous-bike/data/100k_images/` and `s3://autonomous-bike/data/hybridNets_data.tar`

Live checkpoints and JSON metrics are synced to the same `MODEL_OUTPUT_PATH` during training through SageMaker checkpointing. The final SageMaker model artifact is still written at job completion.


In [42]:
from pathlib import Path
from urllib.parse import urlparse
import json

import boto3
import sagemaker
from botocore.exceptions import ClientError
from sagemaker.pytorch import PyTorch

In [43]:
# Static configuration
REGION = "us-east-1"
ROLE_ARN = "arn:aws:iam::253490779227:role/service-role/AmazonSageMakerAdminIAMExecutionRole_1"
BUCKET = "autonomous-bike"

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "TrainingScript.py").exists(), (
    "Run this notebook from /home/aman/Projects/Auto/HybridNets so SageMaker can bundle the repo correctly."
)

RUN_MODE = "real"  # change to "real" for the full training run
SMOKE_S3_IMAGES = f"s3://{BUCKET}/data/hybridnets_smoke/images/"
SMOKE_S3_HYBRIDNETS_DATA = f"s3://{BUCKET}/data/hybridnets_smoke/hybridNets_data_smoke.tar"
REAL_S3_IMAGES = f"s3://{BUCKET}/data/100k_images/"
REAL_S3_HYBRIDNETS_DATA = f"s3://{BUCKET}/data/hybridNets_data.tar"
EXPECTED_IMAGE_COUNTS_BY_MODE = {
    "smoke": {"train": 64, "val": 16},
    "real": {"train": 70000, "val": 10000},
}
HYPERPARAMETERS_BY_MODE = {
    "smoke": REPO_ROOT / "sagemaker_smoketest_hyperparameters.json",
    "real": REPO_ROOT / "sagemaker_real_hyperparameters.json",
}

if RUN_MODE not in {"smoke", "real"}:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

if RUN_MODE == "smoke":
    S3_IMAGES = SMOKE_S3_IMAGES
    S3_HYBRIDNETS_DATA = SMOKE_S3_HYBRIDNETS_DATA
    BASE_JOB_NAME = "hybridnets-road-only-smoke"
else:
    S3_IMAGES = REAL_S3_IMAGES
    S3_HYBRIDNETS_DATA = REAL_S3_HYBRIDNETS_DATA
    BASE_JOB_NAME = "hybridnets-road-only"

EXPECTED_IMAGE_COUNTS = EXPECTED_IMAGE_COUNTS_BY_MODE[RUN_MODE]
HYPERPARAMETERS_PATH = HYPERPARAMETERS_BY_MODE[RUN_MODE]

INSTANCE_TYPE = "ml.g6e.2xlarge"
INSTANCE_COUNT = 1
VOLUME_SIZE_GB = 200
MAX_RUN_SECONDS = 172800

MODEL_OUTPUT_PATH = f"s3://{BUCKET}/hybridnets/model-artifacts"
METRICS_OUTPUT_PATH = f"s3://{BUCKET}/hybridnets/metrics"
CHECKPOINT_LOCAL_PATH = "/opt/ml/checkpoints"
CHECKPOINT_S3_URI = MODEL_OUTPUT_PATH

with open(HYPERPARAMETERS_PATH, "r", encoding="utf-8") as f:
    HYPERPARAMETERS = json.load(f)

print(f"Run mode: {RUN_MODE}")
print(f"Repo root: {REPO_ROOT}")
print(f"Bucket: {BUCKET}")
print(f"Images input: {S3_IMAGES}")
print(f"Bundle input: {S3_HYBRIDNETS_DATA}")
print(f"Hyperparameters: {HYPERPARAMETERS_PATH}")
print(f"Checkpoint S3 URI: {CHECKPOINT_S3_URI}")

Run mode: real
Repo root: /home/aman/Projects/Auto/HybridNets
Bucket: autonomous-bike
Images input: s3://autonomous-bike/data/100k_images/
Bundle input: s3://autonomous-bike/data/hybridNets_data.tar
Hyperparameters: /home/aman/Projects/Auto/HybridNets/sagemaker_real_hyperparameters.json
Checkpoint S3 URI: s3://autonomous-bike/hybridnets/model-artifacts


In [44]:
# AWS / SageMaker session helpers
boto_session = boto3.Session(region_name=REGION)
s3_client = boto_session.client("s3")
sagemaker_session = sagemaker.Session(boto_session=boto_session)


def split_s3_url(s3_url: str):
    parsed = urlparse(s3_url)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Invalid S3 URL: {s3_url}")
    return parsed.netloc, parsed.path.lstrip("/")


def count_s3_objects(bucket: str, prefix: str):
    prefix = prefix.rstrip("/") + "/"
    paginator = s3_client.get_paginator("list_objects_v2")
    object_count = 0
    total_size = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            key = item["Key"]
            if key.endswith("/"):
                continue
            object_count += 1
            total_size += item["Size"]
    return object_count, total_size


def bytes_to_gb(num_bytes: int):
    return round(num_bytes / (1024 ** 3), 2)


def inspect_image_channel(s3_root: str):
    result = {
        "channel": "images",
        "root": s3_root,
        "status": "ready",
        "train_count": 0,
        "val_count": 0,
        "train_size_gb": 0.0,
        "val_size_gb": 0.0,
        "notes": [],
    }
    bucket, prefix = split_s3_url(s3_root)
    train_count, train_size = count_s3_objects(bucket, f"{prefix.rstrip('/')}/train")
    val_count, val_size = count_s3_objects(bucket, f"{prefix.rstrip('/')}/val")
    result.update(
        {
            "train_count": train_count,
            "val_count": val_count,
            "train_size_gb": bytes_to_gb(train_size),
            "val_size_gb": bytes_to_gb(val_size),
        }
    )
    if train_count == 0 or val_count == 0:
        result["status"] = "missing_split"
        result["notes"].append("Missing train/ or val/ objects under this root.")
    if train_count != EXPECTED_IMAGE_COUNTS["train"]:
        result["status"] = "count_mismatch"
        result["notes"].append(
            f"Expected {EXPECTED_IMAGE_COUNTS['train']} train files, found {train_count}."
        )
    if val_count != EXPECTED_IMAGE_COUNTS["val"]:
        result["status"] = "count_mismatch"
        result["notes"].append(
            f"Expected {EXPECTED_IMAGE_COUNTS['val']} val files, found {val_count}."
        )
    return result


def inspect_bundle_object(s3_url: str):
    bucket, key = split_s3_url(s3_url)
    result = {
        "channel": "hybridnets_data",
        "root": s3_url,
        "status": "ready",
        "size_gb": 0.0,
        "notes": [],
    }
    try:
        response = s3_client.head_object(Bucket=bucket, Key=key)
    except ClientError as exc:
        result["status"] = "missing_object"
        result["notes"].append(str(exc))
        return result
    result["size_gb"] = bytes_to_gb(response["ContentLength"])
    return result

In [45]:
# Live S3 preflight for the road-only trainer
channel_inventory = [
    inspect_image_channel(S3_IMAGES),
    inspect_bundle_object(S3_HYBRIDNETS_DATA),
]

for item in channel_inventory:
    print(json.dumps(item, indent=2))
    print("-" * 80)

blocking_channels = [item for item in channel_inventory if item["status"] != "ready"]
if blocking_channels:
    print("Preflight blocked. Fix the channels above before calling estimator.fit(...).")
else:
    print("Both road-only HybridNets channels are ready.")

{
  "channel": "images",
  "root": "s3://autonomous-bike/data/100k_images/",
  "status": "ready",
  "train_count": 70000,
  "val_count": 10000,
  "train_size_gb": 3.78,
  "val_size_gb": 0.54,
  "notes": []
}
--------------------------------------------------------------------------------
{
  "channel": "hybridnets_data",
  "root": "s3://autonomous-bike/data/hybridNets_data.tar",
  "status": "ready",
  "size_gb": 2.67,
  "notes": []
}
--------------------------------------------------------------------------------
Both road-only HybridNets channels are ready.


In [46]:
# Estimator configuration
required_repo_files = [
    "TrainingScript.py",
    "backbone.py",
    "requirements.txt",
    "hybridnets/model.py",
    "hybridnets/loss.py",
    "utils/utils.py",
    "encoders/timm_efficientnet.py",
]
missing_repo_files = [
    rel_path for rel_path in required_repo_files if not (REPO_ROOT / rel_path).exists()
]
if missing_repo_files:
    raise FileNotFoundError(
        "Missing HybridNets source files required for SageMaker packaging: "
        + ", ".join(missing_repo_files)
    )

estimator = PyTorch(
    entry_point="TrainingScript.py",
    source_dir=str(REPO_ROOT),
    role=ROLE_ARN,
    framework_version="2.4",
    py_version="py311",
    instance_count=INSTANCE_COUNT,
    instance_type=INSTANCE_TYPE,
    volume_size=VOLUME_SIZE_GB,
    max_run=MAX_RUN_SECONDS,
    output_path=MODEL_OUTPUT_PATH,
    code_location=MODEL_OUTPUT_PATH,
    output_data_config={"S3OutputPath": METRICS_OUTPUT_PATH},
    checkpoint_s3_uri=CHECKPOINT_S3_URI,
    checkpoint_local_path=CHECKPOINT_LOCAL_PATH,
    hyperparameters=HYPERPARAMETERS,
    environment={
        "PYTHONUNBUFFERED": "1",
        "MPLCONFIGDIR": "/tmp/matplotlib",
    },
    base_job_name=BASE_JOB_NAME,
    sagemaker_session=sagemaker_session,
)

print("Estimator configured:")
print("  entry_point: TrainingScript.py")
print(f"  source_dir: {REPO_ROOT}")
print(f"  instance: {INSTANCE_TYPE}")
print(f"  images: {S3_IMAGES}")
print(f"  hybridnets_data: {S3_HYBRIDNETS_DATA}")
print(f"  model output: {MODEL_OUTPUT_PATH}")
print(f"  metrics output: {METRICS_OUTPUT_PATH}")
print(f"  checkpoint local path: {CHECKPOINT_LOCAL_PATH}")
print(f"  checkpoint s3 uri: {CHECKPOINT_S3_URI}")

Estimator configured:
  entry_point: TrainingScript.py
  source_dir: /home/aman/Projects/Auto/HybridNets
  instance: ml.g6e.2xlarge
  images: s3://autonomous-bike/data/100k_images/
  hybridnets_data: s3://autonomous-bike/data/hybridNets_data.tar
  model output: s3://autonomous-bike/hybridnets/model-artifacts
  metrics output: s3://autonomous-bike/hybridnets/metrics
  checkpoint local path: /opt/ml/checkpoints
  checkpoint s3 uri: s3://autonomous-bike/hybridnets/model-artifacts


In [47]:
# Launch training only when the two-channel road-only preflight passes.
if blocking_channels:
    raise RuntimeError(
        "Refusing to launch HybridNets training because the road-only S3 preflight is not clean. "
        "Fix the image prefix or the hybridNets_data.tar object, rerun the preflight cell, then rerun this cell."
    )

training_inputs = {
    "images": S3_IMAGES,
    "hybridnets_data": S3_HYBRIDNETS_DATA,
}

estimator.fit(training_inputs, wait=True, logs="All")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.estimator:SMDebug Does Not Currently Support                         Distributed Training Jobs With Checkpointing Enabled
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: hybridnets-road-only-2026-04-20-01-37-01-716


2026-04-20 01:43:12 Starting - Starting the training job
2026-04-20 01:43:12 Pending - Training job waiting for capacity......
2026-04-20 01:44:21 Pending - Preparing the instances for training......
2026-04-20 01:44:58 Downloading - Downloading input data...............
2026-04-20 01:47:35 Downloading - Downloading the training image.......bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.127.08
Current installed NVIDIA driver version is 570.211.01
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-04-20 01:49:14,315 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-20 01:49:14,334 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-20 01:49:14,343 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-

In [ ]:
!python InferRoadVideo.py \ --weights /home/aman/Projects/Auto/models/hybridNet/model.pth \ --video /home/aman/Projects/Auto/test_videos/mp4Videos/IMG_5098.mp4 \ --output /home/aman/Projects/Auto/test_videos/mp4Videos/IMG_5098_hybridnets_640x384.mp4 \ --image-width 640 \ --image-height 384 \ --output-width 640 \ --output-height 384 \ --device cuda \ --gpu-ids 0,1 \ --batch-size 8 \ -amp true
